In [1]:
import networkx as nx
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import gzip
import csv
from pies import pies_sampling
from ties import ties_sampling
from scipy.stats import ks_2samp
random.seed(42)
np.random.seed(42)
# url https://www.researchgate.net/publication/254639513_Network_Sampling_via_Edge-based_Node_Selection_with_Graph_Induction

In [2]:
# demo data https://snap.stanford.edu/data/soc-sign-bitcoin-otc.html
filename = 'data/soc-sign-bitcoinotc.csv.gz'
data = []
with gzip.open(filename, 'rt') as f:
    for row in csv.reader(f):
        data.append(row)
data = np.array(data)
df = pd.DataFrame(data,columns=['source','target','rating','time'])
G = nx.DiGraph()
G.add_edges_from(df[['source','target']].values)
# add time and rating to the edges
for i in range(len(df)):
    G[df['source'][i]][df['target'][i]]['time'] = df['time'][i]
    G[df['source'][i]][df['target'][i]]['rating'] = df['rating'][i]

In [3]:
# benchmarks van de paper sectie 3.2

# • S1: In-degree distribution: for every degree d, we count
# the number of nodes with in-degree d. Typically it follows
# a power-law and some other heavy tailed distribution [5].
# • S2: Out-degree distribution.
# • S3: The distribution of sizes of weakly connected components (“wcc”): a set of nodes is weakly connected if for
# any pair of nodes u and v there exists an undirected path
# from u to v.
# • S4: The distribution of sizes of strongly connected
# components (“scc”): a set of nodes is strongly connected, if
# for any pair of nodes u and v, there exists a directed path
# from u to v and from v to u.
# • S5: Hop-plot: the number P(h) of reachable pairs of
# nodes at distance h or less; h is the number of hops [11].
# • S6: Hop-plot on the largest WCC.
# • S7: The distribution of the first left singular vector of
# the graph adjacency matrix versus the rank.
# • S8: The distribution of singular values of the graph
# adjacency matrix versus the rank. Spectral properties of
# graphs often follow a heavy-tailed distribution [3].
# • S9: The distribution of the clustering coefficient Cd [16]
# defined as follows. Let node v have k neighbors; then at most
# k(k −1)/2 edges can exist between them. Let Cv denote the
# fraction of these allowable edges that actually exist. Then
# Cd is defined as the average Cv over all nodes v of degree d.

# temporal graph evolution

#  T1: Densification Power Law (DPL) [9]: number of
# edges versus the number of nodes over time. DPL states
# that e(t) ∝ n(t)^a. The densification exponent a is typically
# greater than 1, implying that the average degree of a node
# in the graph is increasing over time.
# • T2: The effective diameter of the graph over time,
# which is defined as the minimum number of hops in which
# 90% of all connected pairs of nodes can reach each other.
# It has been observed that the effective diameter generally
# shrinks or stabilizes as the graph grows with time [9].
# • T3: The normalized size of the largest connected component (CC) over time.
# • T4: The largest singular value of graph adjacency
# matrix over time.
# • T5: Average clustering coefficient C over time [16]: C
# at time t is the average Cv of all nodes v in graph at time t.

In [4]:
class Benchmark:

# add logarithmic bins for all S functions
    def __init__(self,G,Gs):
        self.G = G
        self.Gs = Gs

    def S1(self):
        #  In-degree distribution: for every degree d, we count
        # the number of nodes with in-degree d. Typically it follows
        # a power-law and some other heavy tailed distribution
        # compute using the kolmogorov-smirnov test
        in_degree = dict(self.G.in_degree()) # todo kijken of het beter kan
        in_degree = list(in_degree.values())
        in_degree2 = dict(self.Gs.in_degree())
        in_degree2 = list(in_degree2.values())
        return ks_2samp(in_degree,in_degree2).statistic
    
    def S2(self):
        # Out-degree distribution.
        out_degree = dict(self.G.out_degree())
        out_degree = list(out_degree.values())
        out_degree2 = dict(self.Gs.out_degree())
        out_degree2 = list(out_degree2.values())
        return ks_2samp(out_degree,out_degree2).statistic
    
    def S3(self):
        # S3: The distribution of sizes of weakly connected components (“wcc”): a set of nodes is weakly connected if for
        # any pair of nodes u and v there exists an undirected path
        # from u to v.
        wcc = list(nx.weakly_connected_components(self.G))
        wcc = [len(x) for x in wcc]
        wcc2 = list(nx.weakly_connected_components(self.Gs))
        wcc2 = [len(x) for x in wcc2]
        return ks_2samp(wcc,wcc2).statistic
    def S4(self):
        # • S4: The distribution of sizes of strongly connected
        # components (“scc”): a set of nodes is strongly connected, if
        # for any pair of nodes u and v, there exists a directed path
        # from u to v and from v to u.
        scc = list(nx.strongly_connected_components(self.G))
        scc = [len(x) for x in scc]
        scc2 = list(nx.strongly_connected_components(self.Gs))
        scc2 = [len(x) for x in scc2]
        return ks_2samp(scc,scc2).statistic
    
    def S5(self):
        # • S5: Hop-plot: the number P(h) of reachable pairs of
        # nodes at distance h or less; h is the number of hops [11].
        p_cum1 = self.compute_hopplot(self.G)
        p_cum2 = self.compute_hopplot(self.Gs)
        return ks_2samp(list(p_cum1.values()),list(p_cum2.values())).statistic

    def S6(self):
        # • S6: Hop-plot on the largest WCC.
        largest_wcc = max(nx.weakly_connected_components(self.G), key=len)
        largest_wcc2 = max(nx.weakly_connected_components(self.Gs), key=len)
        p_cum1 = self.compute_hopplot(self.G.subgraph(largest_wcc))
        p_cum2 = self.compute_hopplot(self.Gs.subgraph(largest_wcc2))
        return ks_2samp(list(p_cum1.values()),list(p_cum2.values())).statistic
    
    def S7(self):
        # • S7: The distribution of the first left singular vector of
        # the graph adjacency matrix versus the rank.
        A1 = nx.adjacency_matrix(self.G).toarray()
        A2 = nx.adjacency_matrix(self.Gs).toarray()
        #Perform Singular Value Decomposition (SVD)
        U1, _, _ = np.linalg.svd(A1)
        U2, _, _ = np.linalg.svd(A2)
        first_vector_1 = U1[:, 0]
        first_vector_2 = U2[:, 0]
        return ks_2samp(np.abs(first_vector_1), np.abs(first_vector_2)).statistic

    def S8(self):
         # • S8: The distribution of singular values of the graph
        # adjacency matrix versus the rank. Spectral properties of
        # graphs often follow a heavy-tailed distribution [3].
        # Step 2: Get the adjacency matrices for both graphs

        # TODO optimize with S7
        A1 = nx.adjacency_matrix(self.G).toarray()
        A2 = nx.adjacency_matrix(self.Gs).toarray()
        _, singular_values_1, _ = np.linalg.svd(A1)
        _, singular_values_2, _ = np.linalg.svd(A2)
        sorted_singular_values_1 = np.sort(singular_values_1)[::-1]
        sorted_singular_values_2 = np.sort(singular_values_2)[::-1]
        return ks_2samp(sorted_singular_values_1, sorted_singular_values_2).statistic
    def S9(self):
         # • S9: The distribution of the clustering coefficient Cd [16]
        # defined as follows. Let node v have k neighbors; then at most
        # k(k −1)/2 edges can exist between them. Let Cv denote the
        # fraction of these allowable edges that actually exist. Then
        # Cd is defined as the average Cv over all nodes v of degree d.
        avg_clustering1 = self.degree_clustering_distribution(self.G)
        avg_clustering2 = self.degree_clustering_distribution(self.Gs)
        clustering_G1 = [C for degree, clustering in avg_clustering1.items() for C in [clustering]]
        clustering_G2 = [C for degree, clustering in avg_clustering2.items() for C in [clustering]]
        return ks_2samp(clustering_G1, clustering_G2).statistic

    def compute_hopplot(self,graph):
        # Compute shortest paths
        all_lengths = [
            length for _, paths in nx.all_pairs_shortest_path_length(graph)
            for length in paths.values()
        ]

        # Count hops using NumPy
        unique_hops, counts = np.unique(all_lengths, return_counts=True)

        # Compute cumulative counts
        cumulative_counts = np.cumsum(counts) / (len(graph) * (len(graph) - 1))

        return dict(zip(unique_hops, cumulative_counts))
    
    def degree_clustering_distribution(self,G):
        # Get the clustering coefficients for all nodes in the graph
        clustering_coeffs = nx.clustering(G)
        
        # Group clustering coefficients by degree
        degree_clustering = {}
        
        for node, C_v in clustering_coeffs.items():
            degree = G.degree(node)
            
            if degree not in degree_clustering:
                degree_clustering[degree] = []
            
            degree_clustering[degree].append(C_v)
        
        # Calculate the average clustering coefficient for each degree
        avg_clustering = {degree: np.mean(clustering) for degree, clustering in degree_clustering.items()}
        
        return avg_clustering

In [5]:
Gs = pies_sampling(G,0.1)
benchmark = Benchmark(G,Gs)
print(f"S1: {benchmark.S1()}")
print(f"S2: {benchmark.S2()}")
print(f"S3: {benchmark.S3()}")
print(f"S4: {benchmark.S4()}")
# # print(f"S5: {benchmark.S5()}")
# # print(f"S6: {benchmark.S6()}")
# print(f"S7: {benchmark.S7()}")
# print(f"S8: {benchmark.S8()}")
print(f"S9: {benchmark.S9()}")
# print(f"T1: {benchmark.T1()}")
# print(f"T2: {benchmark.T2()}")
# print(f"T2: {benchmark.T3()}")

S1: 0.12508579288561616
S2: 0.21548117037150077
S3: 0.25
S4: 0.007454160013215132
S9: 0.28879310344827586


In [29]:
benchmark = Benchmark(G,Gs)
Gt = benchmark.make_snapshots_nodes(G,5)
fractions = [len(snapshot.nodes()) / len(G.nodes()) for snapshot in Gt]
St = [ties_sampling(G,fraction) for fraction in fractions]
print([len(snapshot.nodes())/len(G.nodes) for snapshot in St])
print([len(snapshot.nodes())/len(G.nodes) for snapshot in Gt])
print([len(snapshot.edges()) for snapshot in St])
print([len(snapshot.edges()) for snapshot in Gt])
fractions

[0.20013603128719606, 0.400102023465397, 0.6002380547525931, 0.800034007821799, 1.0]
[0.20013603128719606, 0.400102023465397, 0.600068015643598, 0.800034007821799, 1.0]
[14383, 24122, 29824, 33396, 35592]
[6456, 15235, 21968, 30794, 35592]


[0.20013603128719606,
 0.400102023465397,
 0.600068015643598,
 0.800034007821799,
 1.0]

In [30]:
# benchmark them
for gt_t, st_t in zip(Gt, St):
    benchmark = Benchmark(gt_t, st_t)
    print(f"-- New Benchmark --")
    print(f"S1: {benchmark.S1()}")
    print(f"S2: {benchmark.S2()}")
    print(f"S3: {benchmark.S3()}")
    print(f"S4: {benchmark.S4()}")
    # # print(f"S5: {benchmark.S5()}")
    # # print(f"S6: {benchmark.S6()}")
    # print(f"S7: {benchmark.S7()}")
    # print(f"S8: {benchmark.S8()}")
    print(f"S9: {benchmark.S9()}")

-- New Benchmark --
S1: 0.27017841971113
S2: 0.2421410365335599
S3: 1.0
S4: 0.07617567042363001
S9: 0.629673125856332
-- New Benchmark --
S1: 0.21589460263493412
S2: 0.1971950701232469
S3: 1.0
S4: 0.04435055340988621
S9: 0.22611585944919277
-- New Benchmark --
S1: 0.15086715735343817
S2: 0.13586174288794506
S3: 1.0
S4: 0.016002304224635108
S9: 0.09788149092067537
-- New Benchmark --
S1: 0.06822529224229543
S2: 0.056535600425079706
S3: 0.3333333333333333
S4: 0.008029900782885048
S9: 0.07355031021455062
-- New Benchmark --
S1: 0.0
S2: 0.0
S3: 0.0
S4: 0.0
S9: 0.005747126436781609
